# SmallLLM Pathology model


Training a LLM from scratch on famous pathology related textbooks. I created this dataset by combining 13 renouned pathology textbooks suggested by a pathologist and have a total of 52 million characters to train on.  
My aim for making this project is to enhance my understanding on transformers and large language model in general.

### Importing my own dataset from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="Pranjal888/Pathology_textbook_training_dataset",
    filename="pathology_book_combined_data.txt",
    repo_type="dataset"
)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Loaded {len(text):,} characters")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pathology_book_combined_data.txt:   0%|          | 0.00/53.0M [00:00<?, ?B/s]

Loaded 52,746,593 characters


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(''.join(chars))

Vocabulary size: 258
	
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~ ¡¢£¥§©«¬­®°±²³´µ·º»¼½¿ÀÁÄÅÇÉÍÓÖ×ØÜàáâãäåæçèéêëìíïñòóôö÷øùúüþćęğıłńşˆ;Ϳ΀΁ΔΚΝΩαβγδεζθκλμπστψω  ​–—‘’“”„†•…′€™↑→↓−∼≅≈≤≥≫⊥■□►▼●♦✓❖ﬁﬂ


### Cleaning the dataset by removing unnecessary and filler characters

In [ ]:
import re

def clean_text(text):
    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        " \n\t"
        ".,;:!?()-–—'\"/°%+"
        "αβγδεζθκλμπστψω"
        "µ²³±≤≥"
    )
    return ''.join(c if c in allowed else ' ' for c in text)

text = clean_text(text)

text = re.sub(r' +', ' ', text)
text = re.sub(r'\n+', '\n', text)

chars = sorted(set(text))
vocab_size = len(chars)
print(f"New vocabulary size: {vocab_size}")
print(''.join(chars))

New vocabulary size: 103
	
 !"%'()+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz°±²³µαβγδεζθκλμπστψω–—≤≥


In [ ]:
print(text[:100])

General Pathology
Introduction to Pathology
Pathology is literally the study (logos) of suffering (p


### Char to Int mapping

In [ ]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

### Encoding the data dataset

In [ ]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape)
print(data[:100])

torch.Size([52623327])
tensor([33, 57, 66, 57, 70, 53, 64,  2, 42, 53, 72, 60, 67, 64, 67, 59, 77,  1,
        35, 66, 72, 70, 67, 56, 73, 55, 72, 61, 67, 66,  2, 72, 67,  2, 42, 53,
        72, 60, 67, 64, 67, 59, 77,  1, 42, 53, 72, 60, 67, 64, 67, 59, 77,  2,
        61, 71,  2, 64, 61, 72, 57, 70, 53, 64, 64, 77,  2, 72, 60, 57,  2, 71,
        72, 73, 56, 77,  2,  7, 64, 67, 59, 67, 71,  8,  2, 67, 58,  2, 71, 73,
        58, 58, 57, 70, 61, 66, 59,  2,  7, 68])


### Splitting data into training and validation


In [ ]:
n = int(0.9*len(data))
train_set = data[:n]
val_set = data[n:]
print(train_set.shape)
print(val_set.shape)

torch.Size([47360994])
torch.Size([5262333])


In [ ]:
torch.manual_seed(3530)
batch_size = 4
block_size = 10

def get_batch(split):
    data = train_set if split == 'train' else val_set
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 10])
tensor([[ 2,  1, 34, 67, 68, 68, 57,  2, 44, 46],
        [56,  2, 15, 16, 11, 17,  8, 12,  2, 32],
        [70, 66,  2, 71, 61, 65, 61, 64, 53, 70],
        [35, 38, 11, 20, 10,  2, 75, 60, 61, 55]])
targets:
torch.Size([4, 10])
tensor([[ 1, 34, 67, 68, 68, 57,  2, 44, 46, 10],
        [ 2, 15, 16, 11, 17,  8, 12,  2, 32, 61],
        [66,  2, 71, 61, 65, 61, 64, 53, 70,  2],
        [38, 11, 20, 10,  2, 75, 60, 61, 55, 60]])


### looks at one character and guesses the next one
at first we start with a dumb bigram model


In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(3530)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([40, 103])
tensor(5.0655, grad_fn=<NllLossBackward0>)
	2Qε≤JT(λdMSHEGRα—κ-BW7votiP±m.lYR:9;8–RVζh)	!8hB9)βh3ε.u–ytA1°6±S9κ-.lzμS15±ksµτ.uZDzπKWcl.uK?A"!β1x


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

Let's generate some content using the above bigram model first, to see where we are currently

In [ ]:
batch_size = 32
for steps in range(10000):

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.563015937805176


In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

	δ!U brgematiealls e
weaitin re chby panor
tins
ldraparin'EOW
2;85.
mer dowe ( sclus IOveady arecis olemeul. (le dinsplenen, and ans athesemesticereassio whed
o-fisuthesiablpntriguct
h.
—λY, ted thon sen
O-hon infre wy d tingiald stialy ferc
h torwhre ar monturad andizevin Sue aly vipinde dl fioghe 180 thys, tias cin l en sy WF: berino oi–16008; doucal 27-4–aie coninf 2 (Farco ast pr pl MPofis testenthareqOPe,°	KA, ls this 20, f contiarimit ias intatweinscantons. sporonesint tinong mourexpanaraph


It generates completely random letters, but it seems as if its trying to form a sentence although it doesnt make sense

Now let's setup the google drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# -------HYPERPARAMETERS------- #
# These are parameters for a 10.8M parameter model
batch_size = 64      
block_size = 256      
max_iters = 10000      
eval_interval = 500
learning_rate = 3e-4   
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384          
n_head = 6           
n_layer = 6       
dropout = 0.2       
# ----------------------------- #

torch.manual_seed(3530)

# -------IMPORTING THE DATA--------- #
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="Pranjal888/Pathology_textbook_training_dataset",
    filename="pathology_book_combined_data.txt",
    repo_type="dataset"
)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Loaded {len(text):,} characters")
# ---------------------------------- #

# ------DATA CLEANING------- #
import re

def clean_text(text):
    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        " \n\t"
        ".,;:!?()-–—'\"/°%+"
        "αβγδεζθκλμπστψω"
        "µ²³±≤≥"
    )
    return ''.join(c if c in allowed else ' ' for c in text)

text = clean_text(text)

text = re.sub(r' +', ' ', text)
text = re.sub(r'\n+', '\n', text)

chars = sorted(set(text))
vocab_size = len(chars)
print(f"New vocabulary size: {vocab_size}")
# ------------------------------- #

# ---------MAPPING CHARACTERS TO INTEGERS---------- #
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # inp: string out: integer
decode = lambda l: ''.join([itos[i] for i in l]) # inp: integer out: string
# ------------------------------------------------- #

# -------------TRAIN TEST SPLIT-------------- #
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
# ------------------------------------------- #

# -----------DATA LOADING----------- #
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Calculating loss
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
# ------------------------------------ #

# -----------MAIN ARCHITECTURE-------- #
# One attention head
# For each character, it will ask:
# Query -> what am i looking for?
# Key -> What do i contain?
# Value -> What do i give?
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

# Runs multiple heads simultaneously
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# This is a expand-then-shrink mechanism which gives the model room to process complex patterns
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Combines attention and feed forward into one complete unit
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# Putting everything together
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) 
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) 
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb 
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
# ---------------------------------- #


model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # save checkpoint every 1000 iters
    if iter % 1000 == 0:
        torch.save(model.state_dict(), '/content/drive/MyDrive/pathology_gpt.pt')
        print(f"checkpoint saved at iter {iter}")

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


Loaded 52,746,593 characters
New vocabulary size: 103
10.818151 M parameters
step 0: train loss 4.7927, val loss 4.7959
checkpoint saved at iter 0
step 500: train loss 2.3031, val loss 2.2996
step 1000: train loss 1.7630, val loss 1.7489
checkpoint saved at iter 1000
step 1500: train loss 1.5299, val loss 1.5218
step 2000: train loss 1.4182, val loss 1.4232
checkpoint saved at iter 2000
step 2500: train loss 1.3554, val loss 1.3662
step 3000: train loss 1.3043, val loss 1.3192
checkpoint saved at iter 3000
step 3500: train loss 1.2722, val loss 1.2866
step 4000: train loss 1.2468, val loss 1.2656
checkpoint saved at iter 4000
step 4500: train loss 1.2211, val loss 1.2455
step 5000: train loss 1.1972, val loss 1.2178
checkpoint saved at iter 5000
step 5500: train loss 1.1847, val loss 1.2038
step 6000: train loss 1.1696, val loss 1.1920
checkpoint saved at iter 6000
step 6500: train loss 1.1514, val loss 1.1775
step 7000: train loss 1.1357, val loss 1.1604
checkpoint saved at iter 7000


This is the output generated by the model

In [ ]:
def generate_text(prompt, max_new_tokens=500):
    # encode the prompt
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    # generate
    generated = decode(m.generate(context, max_new_tokens=max_new_tokens)[0].tolist())
    return generated

print(generate_text("squamous cell carcinoma is characterized by"))
print("---")
print(generate_text("the tumor shows signs of"))
print("---")
print(generate_text("diagnosis of lymphoma requires"))

squamous cell carcinoma is characterized by a released vascular cell in assemblin
dilaty stained hyperesability and muscle fibres, correlates
with odies, sized permission of the system hyper normal supersons
or phosphate hair stain.
Brain
Genetic
fibres.
Signification
Ppercentaging of tunnel copposes
The issue damage encountered by
diepending
noton significantly
helps in a parampbed brain tumour with the brain. The purkisma causes a poor 
prognosis of the malignant fibrosis involving the body veola. The literation of 
Leprital tube that 
---
the tumor shows signs of the duct spleen. About
200% of cases formation showed to
highlight, average givis,
particularly transcriptional ulcer havix, blebostominal splean-
what had improvement complicating features (Fig. 16.26).
These choris approach to blind the liver in spleen is most
common unchard conditions have a distinction of burns—block
in blood and
rupture (Figure 11.77).These direct sord is not known, primarily to have
an important progn

Constraints:
1. The current model is trained on 52 million characters of pathology textbook content and it is not enough to train a proper model.
2. The current model I have can produce accurate medical terms but cannot assess the context because its only a 10.8M parameter model.
3. I tried training a 25M parameter model but i couldnt get a better output because 25M is still very low.

Note:
1. This model was trained on Google Collab using a single T4 GPU and took approximately 1.5hrs
2. The 25M parameter model was trained on Kaggle with 2xT4 GPUs and took me 11hrs. Since I lost the trained model after I refreshed the page, I couldn't download the 25M parameter model.